# Data Integration and Role Tables

---

The notebook:
- uses each database's clean tables: `mpv.parquet` (1_1), `tracts.gpkg` (1_2), `acs.parquet` (1_3)
- **integrates MPV, TIGER, and ACS**:
    - **per tract**: adds TIGER's land area to the ACS tract table and computes `pop_density` and `race_segregation`
    - **per death**: places each death in its tract and gives it that tract's ACS numbers
- **builds** the five role tables:
  1. **the indicators**: the situation in which each death happened, used in clustering
  2. **the descriptors**: who died and where, used to describe the classes
  3. **the benchmarks**: the residents' shares around the deaths (local) and the two tract quintiles, by poverty and race segregation (national)
  4. **the markers**: the variables each slice is read through
  5. **the comparison slicer**: the accountability variable that divides the raw deaths directly
- <span style="color:red">**saves**</span> the role tables + their CSV twins in `data/2_join_and_roles/`:
    - `indicators.parquet`
    - `descriptors.parquet`
    - `benchmarks.parquet`
    - `markers.parquet`
    - `slicer.parquet`


### Imports, setup and inputs

---
- Loads DP05's age bands from `shared/config.py`
- Reads the clean tables the three data notebooks saved: `mpv.parquet`, `tracts.gpkg`, `acs.parquet`

<div style="border:1px solid #999; padding:8px">
The columns for all five role tables are hand-selected and stored in <code>data/2_join_and_roles/handmade/join_roles_select_col.csv</code>.
</div>

In [1]:
import pandas as pd
import geopandas as gpd

from shared.config import AGE_BANDS, AGE_EDGES
from shared.paths import (
    MPV_DIR,
    ACS_DIR,
    TIGER_DIR,
    JOIN_ROLES_DIR,
    JOIN_ROLES_HANDMADE_DIR,
)
from shared.io_utils import write_parquet

mpv = pd.read_parquet(MPV_DIR / "mpv.parquet")
tracts = gpd.read_file(TIGER_DIR / "tracts.gpkg")
acs = pd.read_parquet(ACS_DIR / "acs.parquet")
selected = pd.read_csv(JOIN_ROLES_HANDMADE_DIR / "join_roles_select_col.csv")

### Integrating MPV, TIGER, and ACS

---
Performed in two steps.

#### Per tract

<hr style="border:none; border-top:0.5px solid #999">

This step adds:
- TIGER's land area (`tract_land_area`), as provided by `1_2_tiger.ipynb` in `tracts.gpkg`, to the ACS tract table

It computes and adds:
- **`pop_density`**
- **`race_segregation`:** it shows whether a tract's residents are mostly White, mostly Black, or neither (the Index of Concentration at the Extremes for race, ICE; Krieger et al., 2016)

In [2]:
acs_tiger = acs.merge(tracts[["tract_id", "tract_land_area"]], on="tract_id", how="left")

total_pop = acs_tiger["total_pop"].where(acs_tiger["total_pop"] > 0)
land_km2 = acs_tiger["tract_land_area"] / 1_000_000  # m² -> km²
acs_tiger["pop_density"] = total_pop / land_km2

# Krieger et al. (2016): (non-Hispanic White - non-Hispanic Black) / all residents; the shares are in %
acs_tiger["race_segregation"] = (acs_tiger["pct_white"] - acs_tiger["pct_black"]) / 100

#### Per death

<hr style="border:none; border-top:0.5px solid #999">

A **GeoDataFrame** is needed to join each death's location (`longitude`, `latitude`) from MPV with its tract.

In [3]:
locations = gpd.points_from_xy(mpv["longitude"], mpv["latitude"])
mpv_gdf = gpd.GeoDataFrame(mpv, geometry=locations, crs=tracts.crs)

mpv_tiger = mpv_gdf.sjoin(tracts[["tract_id", "geometry"]], how="left", predicate="within")

mpv_tiger_acs = mpv_tiger.merge(acs_tiger, on="tract_id", how="left")

### Building the five role tables

---

#### The indicators

<hr style="border:none; border-top:0.5px solid #999">

Used in clustering. They describe the **situation** in which a death happened.

**Design choices:**
- **The situation only:** the clustering sees what brought police and the person together
- **Conditional independence:** LCA needs indicators that are independent within each class (Morin et al., 2025), so:
  - **`weapon` is empty for every death not armed:** `weapon` only adds which weapon among the armed, and being armed is counted count once
  - **`less_lethal_also` is a descriptor:** the four force columns would give it away
- **`mpv_id` and `date`:** kept for joins and checks, not clustered

In [4]:
indicator_cols = selected[selected["role"] == "indicator"].set_index("column")["renamed"].to_dict()

indicators = mpv[["mpv_id", "date", *indicator_cols]].rename(columns=indicator_cols)

indicators["weapon"] = indicators["weapon"].where(indicators["armed"] == "armed")

write_parquet(indicators, JOIN_ROLES_DIR / "indicators.parquet")

#### The descriptors

<hr style="border:none; border-top:0.5px solid #999">

**Who** died and **where**, used to describe the LCA's classes: age, sex, race, less-lethal also, poverty, population density.

Each death's age is put into one of the Census's 13 age bands (DP05) **`age_band`**, so the two can be compared.

In [5]:
descriptor_cols = selected.loc[selected["role"] == "descriptor", "column"]

age_band = pd.cut(
    mpv_tiger_acs["age"],
    bins=AGE_EDGES,
    labels=AGE_BANDS,
    right=False,  # each band starts at its edge, the way the names read: "25-34" starts at 25 (without it, 25 would end "20-24")
)
mpv_tiger_acs["age_band"] = age_band.astype(object)  # the band names as plain text

descriptors = mpv_tiger_acs[["mpv_id", "tract_id", *descriptor_cols]]
write_parquet(descriptors, JOIN_ROLES_DIR / "descriptors.parquet")

#### The benchmarks

<hr style="border:none; border-top:0.5px solid #999">

The share of deaths is read through the shares of the residents it is compared with: 1 means the deaths fall exactly as the residents live, above 1 more often, below 1 less often.
- **Local:** compares the people killed with who lives there
- **National (tract quintiles):**
  - tracts are sorted by poverty or race segregation, lowest value first
  - each sorted list is further cut into five groups
  - poverty and the ICE for race are both among Feldman et al.'s (2019) quintile measures
  - this study diverges from their method by weighing each tract by its total population (`total_pop`), so each group holds a fifth of the US population rather than a fifth of the tracts
  - this asks whether the deaths fall in the poorest fifth (quintile 5 for poverty) more often than a fifth of the time

In [6]:
# the ACS variables are renamed (`pct_black` becomes `race = black`)
local_cols = selected[selected["benchmark"] == "local"].set_index("column")["renamed"].to_dict()
local = acs_tiger[["tract_id", *local_cols]].rename(columns=local_cols)

national_cols = selected.loc[selected["benchmark"] == "national", "column"]
national = acs_tiger[["tract_id", "total_pop", *national_cols]]

for measure in national_cols:
    national_with_entry = national[national[measure].notna()]
    # equal values keep tract id order
    national_sorted = national_with_entry.sort_values([measure, "tract_id"])
    national_sorted_pop = national_sorted["total_pop"]

    # its people plus the people of every tract before it in the sorted list
    pop_cumulative = national_sorted_pop.cumsum()
    # step back half the tract's population: if the middle is past a cut line, most of its population falls beyond it
    middle = pop_cumulative - national_sorted_pop / 2
    middle_share = middle / national_sorted_pop.sum()
    fifth = pd.cut(middle_share, bins=[0, 0.2, 0.4, 0.6, 0.8, 1], labels=[1, 2, 3, 4, 5]).astype(int)
    # approximately 20%, since tracts aren't split
    fifth_share = pd.Series(national_sorted_pop).groupby(fifth).sum() / national_sorted_pop.sum()

    tract_fifth = pd.Series(fifth, index=national_sorted.index)
    tract_fifth_share = tract_fifth.map(fifth_share)
    # written into `national`, which still has every tract: the tracts without an entry stay empty (Int64)
    national[f"fifth_{measure}"] = tract_fifth.astype("Int64")
    national[f"fifth_share_{measure}"] = tract_fifth_share

# only needed for the cut
national = national.drop(columns=["total_pop", *national_cols])

benchmarks = local.merge(national, on="tract_id")

write_parquet(benchmarks, JOIN_ROLES_DIR / "benchmarks.parquet")

#### The markers

<hr style="border:none; border-top:0.5px solid #999">

The variables each slice is read through.

In [7]:
marker_cols = selected.loc[selected["role"] == "marker", "column"]

markers = mpv_tiger_acs[["mpv_id", *marker_cols]]
write_parquet(markers, JOIN_ROLES_DIR / "markers.parquet")

#### The comparison slicer

<hr style="border:none; border-top:0.5px solid #999">

The accountability variable that divides the raw deaths directly.

In [8]:
slicer_cols = selected.loc[selected["role"] == "slicer", "column"]
slicer = mpv_tiger_acs[["mpv_id", *slicer_cols]]
write_parquet(slicer, JOIN_ROLES_DIR / "slicer.parquet")